# 🌤️ weather-api-python-project
**Fetch, analyze and visualize a 5-day weather forecast using Python**

---

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

print('✅ Libraries loaded!')

In [ ]:
# ── Settings — change these! ──────────────────────────────────────────
API_KEY = 'PASTE_YOUR_API_KEY_HERE'   # https://openweathermap.org/api
CITY    = 'Algiers'                   # any city in the world
UNITS   = 'metric'                    # metric = °C  |  imperial = °F

UNIT_SYMBOL = '°C' if UNITS == 'metric' else '°F'

In [ ]:
# ── Fetch data ────────────────────────────────────────────────────────
url = 'https://api.openweathermap.org/data/2.5/forecast'
params = {'q': CITY, 'appid': API_KEY, 'units': UNITS, 'cnt': 40}

response = requests.get(url, params=params)

if response.status_code == 200:
    raw = response.json()
    city_name = raw['city']['name']
    country   = raw['city']['country']
    print(f'✅ {city_name}, {country} — {len(raw["list"])} entries fetched')
else:
    print(f'❌ Error {response.status_code}: {response.json().get("message")}')
    print('   → Check your API_KEY and CITY spelling')

In [ ]:
# ── Build DataFrame ───────────────────────────────────────────────────
rows = []
for entry in raw['list']:
    rows.append({
        'datetime'    : entry['dt_txt'],
        'temperature' : entry['main']['temp'],
        'feels_like'  : entry['main']['feels_like'],
        'temp_min'    : entry['main']['temp_min'],
        'temp_max'    : entry['main']['temp_max'],
        'humidity'    : entry['main']['humidity'],
        'wind_speed'  : entry['wind']['speed'],
        'description' : entry['weather'][0]['description'].title(),
        'rain_mm'     : entry.get('rain', {}).get('3h', 0.0),
        'rain_chance' : round(entry.get('pop', 0) * 100),
    })

df = pd.DataFrame(rows)
df['datetime'] = pd.to_datetime(df['datetime'])

print(f'DataFrame: {df.shape[0]} rows x {df.shape[1]} columns')
df.head(5)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────
hottest  = df.loc[df['temperature'].idxmax()]
coldest  = df.loc[df['temperature'].idxmin()]
rainiest = df.loc[df['rain_chance'].idxmax()]

print(f'📍 {city_name}, {country}')
print(f'  Avg temp    : {df["temperature"].mean():.1f} {UNIT_SYMBOL}')
print(f'  Hottest     : {hottest["temperature"]:.1f} {UNIT_SYMBOL} — {hottest["datetime"].strftime("%a %d %b %H:%M")}')
print(f'  Coldest     : {coldest["temperature"]:.1f} {UNIT_SYMBOL} — {coldest["datetime"].strftime("%a %d %b %H:%M")}')
print(f'  Avg humidity: {df["humidity"].mean():.0f}%')
print(f'  Max wind    : {df["wind_speed"].max():.1f} m/s')
print(f'  Rain chance : up to {rainiest["rain_chance"]}% on {rainiest["datetime"].strftime("%a %d %b")}')

In [ ]:
# ── Daily table ───────────────────────────────────────────────────────
daily = df.groupby(df['datetime'].dt.date).agg(
    Min_Temp    = ('temp_min',    'min'),
    Max_Temp    = ('temp_max',    'max'),
    Avg_Temp    = ('temperature', 'mean'),
    Humidity    = ('humidity',    'mean'),
    Rain_Chance = ('rain_chance', 'max'),
).round(1)
daily.index.name = 'Date'
daily

In [ ]:
# ── Save CSV ──────────────────────────────────────────────────────────
city_slug = CITY.lower().replace(' ', '_')
df.to_csv(f'{city_slug}_forecast.csv', index=False)
daily.to_csv(f'{city_slug}_daily.csv')
print(f'✅ Saved {city_slug}_forecast.csv  &  {city_slug}_daily.csv')

In [ ]:
# ── Charts ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle(f'{city_name}, {country} — 5-Day Forecast', fontsize=14, y=1.01)

# Temperature
axes[0].plot(df['datetime'], df['temperature'], color='tomato', lw=2, label=f'Temp ({UNIT_SYMBOL})')
axes[0].plot(df['datetime'], df['feels_like'], color='steelblue', lw=1.5, linestyle='--', label='Feels like')
axes[0].fill_between(df['datetime'], df['temp_min'], df['temp_max'], alpha=0.1, color='tomato')
axes[0].set_ylabel(f'Temp ({UNIT_SYMBOL})')
axes[0].legend(fontsize=9)

# Humidity
axes[1].plot(df['datetime'], df['humidity'], color='mediumseagreen', lw=2)
axes[1].fill_between(df['datetime'], df['humidity'], alpha=0.15, color='mediumseagreen')
axes[1].axhline(70, color='red', linestyle=':', alpha=0.5, label='70% threshold')
axes[1].set_ylabel('Humidity (%)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)

# Rain chance
axes[2].bar(df['datetime'], df['rain_chance'], width=0.1, color='cornflowerblue', alpha=0.85)
axes[2].set_ylabel('Rain chance (%)')
axes[2].set_ylim(0, 100)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.savefig(f'{city_slug}_charts.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Saved {city_slug}_charts.png')